# Lesson 03 - Agentic Design Patterns

У овој лекцији истражујемо три основна образца дизајна за изградњу ефикасних AI агената:

1. **Јасна упутства за агента** — Креирање прецизних, улогу дефинишућих упита који воде понашање агента  
2. **Структурирани излаз са Pydantic моделима** — Осигуравање да агенти враћају предвидљиве, верификоване податке  
3. **Агенти са једним одговорношћу** — Дизајнирање фокусираног агента који сваки ради по једну ствар добро

Примењиваћемо сваки образац на сценарио **рекомандера туристичких дестинација**, постепено Grade развијајући систем који може да препоручи дестинације, провери доступност и уреди логистику.


## Постављање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Образац 1: Јасна упутства за агента

Најемотивнији образац је и најједноставнији: писање јасних, детаљних упутстава за вашег агента.

Добра упутства дефинишу:
- **Ко** је агент (персона и тон)
- **Шта** треба да ради (одговорности корак по корак)
- **Како** треба да се понаша (ограничења и стил)

Испод, креирамо агента за туристичког консијержа са експлицитним упутствима која обликују сваки одговор који производи.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Образац 2: Структурирани излаз са Пидантил моделима

Текст слободног формата је користан за разговор, али системи у наставку рада захтевају структуриране податке.
Комбиновањем **Пидантил модела** са **алатном функцијом**, можемо:

- Дефинисати тачну шему за излаз агента
- Аутоматски верификовати одговоре
- Поуздано интегрисати резултате агента у логику апликације

Такође уводимо алатку која враћа детаље о одредишту како би агент темељио своје препоруке на стварним подацима.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Патерн 3: Агенти са појединачном одговорношћу

Сложени задаци имају користи од поделе рада између више фокусираних агената, од којих сваки има појединачну одговорност:

- **Стручњак за дестинације** који зна о местима и доступности
- **Планирање логистике** који се бави летовима, хотелима и плановима путовања

Ово одражава принцип инжењеринга софтвера *раздвајања брига* — сваки агент је лакши за тестирање, одржавање и независно унапређење.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Резиме

У овој лекцији применили смо три агенцијска дизајн обрасца на сценарио препорука за путовања:

| Образац | Кључна идеја | Добит |
|---|---|---|
| **Јасна упутства** | Дефинишите персону, одговорности и ограничења унапред | Конзистентно, у складу са брендом понашање агента |
| **Структурисани излаз** | Користите Pydantic моделе као формат одговора | Валидацијa, резултати читљиви за машину |
| **Једна одговорност** | Доделите сваком агенту један фокусиран задатак | Једноставније тестирање, одржавање и композиција |

Ови обрасци се природно слажу — можете комбиновати јасна упутства са структурисаним излазом унутар агента са једном одговорношћу да изградите робусне, спремне за продукцију системе.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Одрицање од одговорности**:
Овај документ је преведен помоћу АИ преводилачке услуге [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да обезбедимо тачност, молимо вас да имате у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом матичном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из употребе овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
